# YOLOv8n PCB Defect Detection -- Fine-tuning

Fine-tunes YOLOv8n on the PCB defect dataset (6 classes: Missing Hole, Mouse Bite,
Open Circuit, Short, Spur, Spurious Copper). Dataset is re-split 80/10/10
(train/val/test) with a fixed seed, discarding any pre-existing split.

Reference: Redmon et al. (2016) "You Only Look Once: Unified, Real-Time Object Detection"
https://arxiv.org/abs/1506.02640

## 1. Install and imports

In [ ]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import yaml
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_DIR   = Path("/kaggle/input/pcb-defect-dataset")
WORK_DIR    = Path("/kaggle/working")
DATASET_DIR = WORK_DIR / "pcb_dataset"
RUNS_DIR    = WORK_DIR / "runs"
PLOTS_DIR   = WORK_DIR / "plots"

for d in [DATASET_DIR, RUNS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Ultralytics version:", __import__("ultralytics").__version__)
print("Input dir contents:")
for p in sorted(INPUT_DIR.rglob("*"))[:30]:
    print(" ", p.relative_to(INPUT_DIR))

## 2. Inspect dataset structure

In [ ]:
# Count images and labels to understand the layout before re-splitting.
images = sorted(INPUT_DIR.rglob("*.jpg")) + sorted(INPUT_DIR.rglob("*.png")) + sorted(INPUT_DIR.rglob("*.JPG"))
labels = sorted(INPUT_DIR.rglob("*.txt"))

# Filter out any non-annotation txt files (e.g. classes.txt, README).
labels = [l for l in labels if l.stem != "classes" and l.stem != "README"]

print(f"Total images found: {len(images)}")
print(f"Total label files found: {len(labels)}")
print()

# Show unique class ids present in labels.
class_ids_found = set()
for lf in labels[:500]:  # sample first 500 for speed
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            class_ids_found.add(int(line.split()[0]))
print(f"Class IDs present (sample): {sorted(class_ids_found)}")

# Show a few label file contents.
print()
print("Sample label file:", labels[0])
print(labels[0].read_text()[:300])

## 3. Match images to labels and re-split 80/10/10

In [ ]:
# Build matched (image, label) pairs. Only keep pairs where both exist.
label_stems = {l.stem: l for l in labels}
pairs = []
for img in images:
    if img.stem in label_stems:
        pairs.append((img, label_stems[img.stem]))

print(f"Matched image-label pairs: {len(pairs)}")
print(f"Images without labels (skipped): {len(images) - len(pairs)}")

# Shuffle and split 80/10/10.
random.shuffle(pairs)
n = len(pairs)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

splits = {
    "train": pairs[:n_train],
    "val":   pairs[n_train:n_train + n_val],
    "test":  pairs[n_train + n_val:],
}

for split, split_pairs in splits.items():
    print(f"{split:>6}: {len(split_pairs)} pairs")

## 4. Copy files into YOLO directory structure

In [ ]:
# YOLO expects:
#   dataset/images/train/*.jpg
#   dataset/labels/train/*.txt
#   (same for val and test)

for split, split_pairs in splits.items():
    img_dir = DATASET_DIR / "images" / split
    lbl_dir = DATASET_DIR / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path, lbl_path in split_pairs:
        shutil.copy(img_path, img_dir / img_path.name)
        shutil.copy(lbl_path, lbl_dir / lbl_path.name)

print("Directory structure created:")
for split in ["train", "val", "test"]:
    n_imgs = len(list((DATASET_DIR / "images" / split).iterdir()))
    n_lbls = len(list((DATASET_DIR / "labels" / split).iterdir()))
    print(f"  {split}: {n_imgs} images, {n_lbls} labels")

## 5. Create dataset.yaml

In [ ]:
# Class names for the PCB defect dataset.
# Adjust if the inspect cell above shows different class ordering.
CLASS_NAMES = [
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]

dataset_yaml = {
    "path":  str(DATASET_DIR),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATASET_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)

print("dataset.yaml written:")
print(yaml_path.read_text())

## 6. Fine-tune YOLOv8n

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    seed=SEED,
    project=str(RUNS_DIR),
    name="pcb_yolov8n",
    exist_ok=True,
    verbose=True,
)

print("Training complete.")
print(f"Best weights: {RUNS_DIR}/pcb_yolov8n/weights/best.pt")

## 7. Evaluate on test set

In [ ]:
best_weights = RUNS_DIR / "pcb_yolov8n" / "weights" / "best.pt"
eval_model   = YOLO(str(best_weights))

metrics = eval_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name="pcb_eval",
    exist_ok=True,
    verbose=True,
)

print()
print("=" * 50)
print("Test set results")
print("=" * 50)
print(f"mAP@0.5:       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")
print(f"Precision:     {metrics.box.mp:.4f}")
print(f"Recall:        {metrics.box.mr:.4f}")
print()
print("Per-class AP@0.5:")
for name, ap in zip(CLASS_NAMES, metrics.box.ap50):
    print(f"  {name:<20} {ap:.4f}")

## 8. Inference on test samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Pick 8 random test images.
test_images = sorted((DATASET_DIR / "images" / "test").iterdir())
random.shuffle(test_images)
sample_images = test_images[:8]

# Run inference.
infer_results = eval_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    iou=0.45,
    verbose=False,
)

# Plot predictions.
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, result, img_path in zip(axes, infer_results, sample_images):
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=7)

    if result.boxes is not None and len(result.boxes):
        img_w, img_h = img.size
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0])
            conf   = float(box.conf[0])
            label  = f"{CLASS_NAMES[cls_id]} {conf:.2f}"

            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=1.5, edgecolor="lime", facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 4, 0), label, color="lime",
                    fontsize=6, backgroundcolor="black")

plt.suptitle("YOLOv8n PCB Defect Detection -- test set inference", fontsize=12)
plt.tight_layout()
plot_path = PLOTS_DIR / "inference_samples.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Inference plot saved to {plot_path}")

## 9. Copy outputs for download

In [ ]:
# Copy best weights, dataset.yaml, and plots to /kaggle/working root.
shutil.copy(best_weights, WORK_DIR / "best_pcb_yolov8n.pt")
shutil.copy(yaml_path, WORK_DIR / "dataset.yaml")
shutil.copy(plot_path, WORK_DIR / "inference_samples.png")

# Copy training curves from runs dir.
curves_src = RUNS_DIR / "pcb_yolov8n" / "results.png"
if curves_src.exists():
    shutil.copy(curves_src, WORK_DIR / "training_curves.png")

print("Files ready for download:")
for f in sorted(WORK_DIR.glob("*.pt")) + sorted(WORK_DIR.glob("*.yaml")) + sorted(WORK_DIR.glob("*.png")):
    size_mb = f.stat().st_size / (1024**2)
    print(f"  {f.name:<40} {size_mb:.1f} MB")